## 2024 07/30 AutoML - Hawkeye Hands

*Last Updated*: 2025-02-24

### Authors
* Nicole Tin


### Overview
This Jupyter notebook is intended to demonstrate ...


### Key Results

- ...

In [2]:
# --- Imports
import shap
import pandas as pd

from pycaret import regression

# import mlflow # creates error


In [5]:
# Dataset
dataset_name = "hawkeye-hands"

# AutoML
experiment_name = "NT_hawkeye-hands-texture-p2"
num_best_models = 5
random_seed = 42

# Paths
root = '/Users/ntin/Documents/DermaML_local/hawkeye-hands-2024-07-29'
# src_dir = '/Users/nicole/Documents/DermaML_local/hawkeye-hands-2024-07-29/processed_images/'
features_dir =  '/Users/ntin/Documents/DermaML_local/hawkeye-hands-2024-07-29/features-2025-02-24/'

image_folder = '/processed_images/'
csv_file = '/metadata.csv'

In [6]:
# -- Read the CSV file
metadata = pd.read_csv(root+ csv_file)
metadata.loc[:, 'Age'] = 2024-metadata['birth_year']
valid_image_fnames_df = pd.DataFrame(metadata.set_index('Age').loc[:, ['right_hand_image_file', 'left_hand_image_file']].stack()).reset_index()
valid_image_fnames_df.columns = ['Age', 'handedness', 'filename']
valid_image_fnames_df.loc[:,'filename'] = valid_image_fnames_df['filename'].apply(lambda x: x[:-5])
valid_image_fnames = valid_image_fnames_df['filename'].to_numpy()

In [20]:
f_dermaml = pd.read_csv(features_dir+'2025-02-24_hawkeye-hands-lbp_glcm-features.csv').drop(columns='Unnamed: 0')
f_redness = pd.read_csv(features_dir+'2025-02-24_hawkeye-hands-redness-features.csv').drop(columns='Unnamed: 0')
f_hessian = pd.read_csv(features_dir+'2025-02-24_hawkeye-hands-hessian-features.csv').drop(columns='Unnamed: 0')

F = f_redness.merge(f_hessian).merge(f_dermaml)
F.loc[:,'filename'] = F['filename'].apply(lambda x:x[:-4])
X = F.join(valid_image_fnames_df.set_index('filename'), on='filename', how='inner').drop(columns=['handedness', 'filename'])
X = X[X['Age'] < 150]
X = X[X['Age'] > 3]

In [29]:
X

,relative_redness_mean,relative_redness_std,skin_folds_hessian,skin_folds_hessian_pct_mask,lbp_0,lbp_1,lbp_2,lbp_3,lbp_4,lbp_5,lbp_6,lbp_7,lbp_8,lbp_9,lbp_10,contrast_scikit,correlation_scikit,energy_scikit,homogeneity_scikit,Age
0,0.783902,3.716256,128516,0.042214,0.022831,0.021934,0.011770,0.010568,0.012823,0.014067,0.012258,0.013762,0.022718,0.777171,0.080097,6.382701,0.992137,0.749812,0.870312,21
1,0.798719,3.144994,112944,0.033554,0.023172,0.023351,0.013589,0.013031,0.018224,0.020246,0.015790,0.016216,0.023628,0.750106,0.082647,7.926946,0.991514,0.723336,0.840914,21
2,1.220194,7.162699,587008,0.169171,0.023416,0.024822,0.014109,0.013595,0.022072,0.023837,0.015792,0.016798,0.024727,0.739501,0.081331,32.170885,0.954015,0.714413,0.769236,40
3,0.835215,4.366527,155375,0.042730,0.025879,0.025666,0.015475,0.014294,0.017623,0.019253,0.016674,0.018092,0.026040,0.730208,0.090796,10.686733,0.990375,0.700761,0.818883,20
4,0.779497,3.809114,96988,0.035910,0.015693,0.016725,0.012868,0.013468,0.019636,0.021435,0.015540,0.015013,0.017421,0.797284,0.054917,6.223111,0.994319,0.777981,0.875967,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,0.731423,3.391351,81140,0.023370,0.022001,0.022475,0.014320,0.014135,0.018464,0.020964,0.017272,0.018007,0.024941,0.745386,0.082034,8.249842,0.993726,0.714947,0.867324,22
572,0.497957,4.100146,142530,0.015985,0.091526,0.074496,0.024219,0.013170,0.010623,0.011175,0.014181,0.026224,0.077530,0.368035,0.288822,79.474882,0.984301,0.269460,0.443510,27
573,0.862216,3.967441,212454,0.048394,0.029628,0.030467,0.019619,0.018546,0.024091,0.025882,0.020803,0.022805,0.031185,0.672450,0.104525,11.837227,0.990300,0.638619,0.774309,22
574,0.974149,4.246954,317438,0.098529,0.026449,0.027916,0.016419,0.018885,0.033596,0.036956,0.023294,0.020803,0.028099,0.676141,0.091442,34.591581,0.962001,0.644298,0.732076,79


In [18]:
# X.to_csv('2024-08-01_NT_Hawkeye-Hands-Texture-Features.csv')
X['Age'].describe()

count    576.000000
mean      46.715278
std       49.094785
min        2.000000
25%       24.000000
50%       37.500000
75%       62.250000
max      789.000000
Name: Age, dtype: float64

In [21]:
metadata.columns

Index(['record_id', 'gender', 'gender_specify', 'birth_year',
       'sex_assigned_at_birth', 'sex_assigned_at_birth_specify',
       'race_ethnicity', 'race_ethnicity_specify', 'ethnicity', 'handedness',
       'occupation', 'driving_time', 'sun_exposure', 'sunscreen_use', 'region',
       'region_specify', 'left_hand_image_file', 'right_hand_image_file',
       'form_complete', 'Age'],
      dtype='object')

1. engineered features only
2. + GLCM wrinkles
3. CNN

preliminary ML model; a lot more interesting things to do be done with features (collaboration) and model improvement;
- describe features
- share model results
- interpretation of graph

In [35]:
# --- Perform AutoML Evaluation

# Set up the dataset for AutoML regression
regression.setup(data=X,
                 target="Age",
                #  log_experiment=True,
                 experiment_name=experiment_name,
                 session_id=random_seed,
                ) 

best_models = regression.compare_models(n_select=num_best_models, verbose=False)


,Description,Value
0,Session id,42
1,Target,Age
2,Target type,Regression
3,Original data shape,"(572, 20)"
4,Transformed data shape,"(572, 20)"
5,Transformed train set shape,"(400, 20)"
6,Transformed test set shape,"(172, 20)"
7,Numeric features,19
8,Preprocess,True
9,Imputation type,simple


In [26]:
regression.pull()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,9.313300e+00,1.675719e+02,1.280880e+01,6.364000e-01,0.2853,2.400000e-01,0.015
rf,Random Forest Regressor,1.010890e+01,1.870198e+02,1.357660e+01,5.921000e-01,0.3049,2.669000e-01,0.085
lightgbm,Light Gradient Boosting Machine,1.000370e+01,1.929672e+02,1.373400e+01,5.822000e-01,0.3040,2.600000e-01,0.163
gbr,Gradient Boosting Regressor,1.039830e+01,1.961244e+02,1.387540e+01,5.733000e-01,0.3082,2.691000e-01,0.016
lr,Linear Regression,1.068120e+01,1.980124e+02,1.395630e+01,5.661000e-01,0.3150,2.780000e-01,0.226
ada,AdaBoost Regressor,1.364530e+01,2.602724e+02,1.609030e+01,4.324000e-01,0.3860,3.999000e-01,0.009
ridge,Ridge Regression,1.523670e+01,3.445103e+02,1.850200e+01,2.504000e-01,0.4135,4.163000e-01,0.004
dt,Decision Tree Regressor,1.297540e+01,3.843270e+02,1.947960e+01,1.487000e-01,0.4251,3.189000e-01,0.004
en,Elastic Net,1.711730e+01,4.246599e+02,2.050960e+01,7.840000e-02,0.4626,4.747000e-01,0.003
lasso,Lasso Regression,1.713530e+01,4.254988e+02,2.052910e+01,7.680000e-02,0.4630,4.751000e-01,0.068


In [33]:
dt = regression.create_model('et')
regression.tune_model(dt, optimize='MAE')
# final_model = regression.finalize_model(best_models[3])
regression.save_experiment('hawkeye-hands-texture-fraction')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,9.3153,160.7642,12.6793,0.6068,0.2686,0.2265
1,10.1810,209.4598,14.4727,0.5906,0.3101,0.2315
2,8.8739,146.0460,12.0850,0.7369,0.2407,0.1866
3,6.7808,90.1893,9.4968,0.7884,0.2634,0.2147
4,9.1408,152.8271,12.3623,0.6327,0.3108,0.2810
5,12.4523,263.0394,16.2185,0.4652,0.3610,0.3243
6,11.7250,226.2461,15.0415,0.5826,0.3101,0.2606
7,7.8284,131.4257,11.4641,0.6801,0.2617,0.2203
8,9.2616,167.0673,12.9255,0.5749,0.2916,0.2642


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,10.8045,179.7155,13.4058,0.5605,0.3094,0.2932
1,11.9309,261.7566,16.1789,0.4884,0.3491,0.2877
2,10.5928,193.5745,13.9131,0.6513,0.2766,0.2349
3,9.1489,129.5068,11.3801,0.6962,0.3124,0.2968
4,10.2958,173.5328,13.1732,0.5830,0.3322,0.3188
5,13.5364,293.9694,17.1455,0.4023,0.3730,0.3460
6,12.8537,254.8168,15.9630,0.5299,0.3239,0.2849
7,10.8658,194.5604,13.9485,0.5264,0.3495,0.3464
8,10.8264,208.6942,14.4463,0.4689,0.3286,0.3137


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [28]:
tuned_models = [regression.tune_model(model, optimize='RMSE') for model in best_models]
ensem_models = [regression.ensemble_model(model, n_estimators=5, optimize='RMSE') for model in tuned_models]
tuned_blend = regression.blend_models(tuned_models, optimize='RMSE')
ensem_blend = regression.blend_models(ensem_models, optimize='RMSE')
model = regression.automl(optimize='RMSE')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,10.8045,179.7155,13.4058,0.5605,0.3094,0.2932
1,11.9309,261.7566,16.1789,0.4884,0.3491,0.2877
2,10.5928,193.5745,13.9131,0.6513,0.2766,0.2349
3,9.1489,129.5068,11.3801,0.6962,0.3124,0.2968
4,10.2958,173.5328,13.1732,0.5830,0.3322,0.3188
5,13.5364,293.9694,17.1455,0.4023,0.3730,0.3460
6,12.8537,254.8168,15.9630,0.5299,0.3239,0.2849
7,10.8658,194.5604,13.9485,0.5264,0.3495,0.3464
8,10.8264,208.6942,14.4463,0.4689,0.3286,0.3137


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,9.8717,175.6621,13.2538,0.5704,0.2899,0.2576
1,11.3001,256.8556,16.0267,0.4980,0.3555,0.2650
2,9.4286,159.1403,12.6151,0.7133,0.2611,0.2136
3,8.3198,115.8712,10.7643,0.7282,0.2860,0.2504
4,9.1219,144.3294,12.0137,0.6531,0.3077,0.2816
5,13.1027,291.5534,17.0749,0.4072,0.3819,0.3458
6,12.6200,252.8267,15.9005,0.5336,0.3414,0.2914
7,8.4131,167.6311,12.9472,0.5920,0.2996,0.2518
8,10.9872,182.5878,13.5125,0.5354,0.3090,0.3062


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,9.5128,141.0248,11.8754,0.6551,0.2594,0.2375
1,11.1401,248.5620,15.7658,0.5142,0.3222,0.2329
2,9.1456,158.6435,12.5954,0.7142,0.2504,0.1912
3,8.6061,132.7871,11.5233,0.6885,0.3187,0.2858
4,10.2854,183.9715,13.5636,0.5579,0.3262,0.2972
5,14.1034,339.8788,18.4358,0.3090,0.3768,0.3542
6,12.7706,272.4765,16.5069,0.4973,0.3572,0.2951
7,7.8312,152.0420,12.3305,0.6299,0.2775,0.2353
8,10.1903,202.8649,14.2431,0.4838,0.3166,0.2925


...

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,9.8359,165.1902,12.8526,0.5960,0.2920,0.2592
1,11.9584,238.0811,15.4299,0.5347,0.3310,0.2807
2,9.6835,170.7970,13.0689,0.6923,0.2500,0.2034
3,7.9021,140.2944,11.8446,0.6709,0.3160,0.2663
4,9.3198,148.0029,12.1656,0.6443,0.3141,0.2897
5,13.0490,272.4901,16.5073,0.4460,0.3537,0.3374
6,12.7970,273.4079,16.5350,0.4956,0.3441,0.2935
7,8.5901,160.8944,12.6844,0.6084,0.3026,0.2597
8,10.3002,210.8311,14.5200,0.4635,0.3268,0.3010


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,9.8595,159.7698,12.6400,0.6093,0.2858,0.2568
1,10.6634,208.0823,14.4251,0.5933,0.3132,0.2514
2,8.9641,148.7856,12.1978,0.7320,0.2417,0.1970
3,7.8169,119.4967,10.9315,0.7197,0.2977,0.2562
4,9.4878,152.0482,12.3308,0.6346,0.3154,0.2925
5,12.6538,251.3836,15.8551,0.4889,0.3543,0.3420
6,12.8164,244.6088,15.6400,0.5488,0.3349,0.2977
7,8.5332,155.8831,12.4853,0.6206,0.3011,0.2567
8,10.9918,209.0080,14.4571,0.4681,0.3287,0.3140


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,8.6385,111.7321,10.5703,0.7267,0.2527,0.2385
1,10.5484,204.4389,14.2982,0.6004,0.3318,0.2381
2,10.7709,214.2428,14.6370,0.6141,0.2754,0.2039
3,9.0831,163.0811,12.7703,0.6174,0.3359,0.3200
4,11.0995,186.2959,13.6490,0.5523,0.3658,0.3181
5,12.2945,256.2015,16.0063,0.4791,0.3441,0.3247
6,11.6550,221.4528,14.8813,0.5915,0.2820,0.2450
7,11.9318,287.4085,16.9531,0.3004,0.3974,0.3752
8,11.1977,187.3407,13.6872,0.5233,0.3143,0.3127


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,9.0028,128.8673,11.3520,0.6848,0.2510,0.2293
1,10.0374,191.9543,13.8548,0.6248,0.2907,0.2245
2,8.6051,142.4935,11.9371,0.7433,0.2315,0.1803
3,6.9723,96.1841,9.8073,0.7743,0.2833,0.2416
4,9.2141,139.0733,11.7929,0.6658,0.3023,0.2795
5,12.6393,256.3628,16.0113,0.4788,0.3487,0.3281
6,11.6174,219.5119,14.8159,0.5951,0.3041,0.2595
7,8.9745,171.7771,13.1064,0.5819,0.3038,0.2651
8,9.5039,171.1606,13.0828,0.5644,0.2964,0.2711


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,9.5987,143.8722,11.9947,0.6481,0.2702,0.2514
1,10.5650,200.8122,14.1708,0.6075,0.3023,0.2446
2,9.0402,150.9409,12.2858,0.7281,0.2360,0.1933
3,7.7393,123.7002,11.1221,0.7098,0.3049,0.2652
4,9.1463,135.4398,11.6379,0.6745,0.3046,0.2838
5,12.3230,241.6720,15.5458,0.5087,0.3431,0.3270
6,12.1522,235.2892,15.3391,0.5659,0.3148,0.2745
7,8.8545,162.9407,12.7648,0.6034,0.3094,0.2693
8,10.1522,186.7235,13.6647,0.5248,0.3138,0.2932


In [ ]:
# final_model = regression.finalize_model(model)
# regression.save_model(final_model, 'hawkeye-hands-texture-fraction')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['relative_redness_mean',
                                              'relative_redness_std',
                                              'GLCM_ASM_Mean_wrinkles_pyfeats',
                                              'GLCM_Contrast_Mean_wrinkles_pyfeats',
                                              'GLCM_Correlation_Mean_wrinkles_pyfeats',
                                              'GLCM_SumOfSquaresVariance_Mean_wrinkles_pyfeats',
                                              'GLCM_InverseDifferenceMoment_Mean_wrinkles_pyf...
                                              'NGTDM_Coarseness_ngtdm',
                                              'NGTDM_Contrast_ngtdm',
                                              'NGTDM_Busyness_ngtdm',
                                              'NGTDM_Complexity_ngtdm',
                                              'NG

In [ ]:
saved_model = regression.load_model('hawkeye-hands-texture-fraction')